# Помесячная сверка Excel vs Озеро (январь–март 2026)

Секции сверки:
1. Количество уникальных клиентов (ИНН).
2. Количество транзакций на клиента.
3. Количество ТСП на клиента.
4. Количество терминалов на клиента.

Транзакции считаются в строгом DAG-периметре:
- `c_trx_class = 'SA'`
- `c_trx_type = 'S01'`
- `c_nter is not null`
- `ods_deleted_flg <> '1'`
- `cf_trx_stat <> 'R'`

In [ ]:
import re
from decimal import Decimal, InvalidOperation
from datetime import datetime
import pandas as pd
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Параметры
base_excel_dir = '/home/jovyan/documents/Equaring/Data'
month_files = {
    '2026-01': '01_Январь_2026.xlsx',
    '2026-02': '02_Февраль_2026.xlsx',
    '2026-03': '03_Марта_2026.xlsx',
}

# Колонки в бизнес-отчетах
excel_inn_col = 'ИНН'
excel_ops_col = 'Количеств операций'
excel_retl_col = 'Ко-во торговых точек'
excel_term_col = 'Кол-во терминалов'

# Размер чанка для IN (...)
chunk_size = 500

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
print('Impala connection initialized')

In [ ]:
def normalize_id(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r"\.0+$", '', s)
    return s if re.fullmatch(r"\d+", s) else None

def normalize_num(v):
    if pd.isna(v):
        return 0
    s = str(v).strip().replace(' ', '').replace('\xa0', '').replace(',', '.')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return 0
    try:
        return float(s)
    except Exception:
        return 0

def to_sql_in_list(vals):
    safe = [str(v).replace("'", "''") for v in vals if pd.notna(v)]
    return ','.join([f"'{x}'" for x in safe])

def split_chunks(items, step=500):
    for i in range(0, len(items), step):
        yield items[i:i+step]

def month_bounds(month_key):
    dt = pd.to_datetime(month_key + '-01')
    start = dt.strftime('%Y-%m-%d')
    end = (dt + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')
    return start, end

## 0) Загрузка и стандартизация Excel по месяцам

In [ ]:
excel_by_month = {}

for month_key, file_name in month_files.items():
    path = f"{base_excel_dir}/{file_name}"
    df = pd.read_excel(path)
    for c in [excel_inn_col, excel_ops_col, excel_retl_col, excel_term_col]:
        if c not in df.columns:
            raise ValueError(f"[{month_key}] в Excel отсутствует колонка: {c}")

    x = pd.DataFrame({
        'inn': df[excel_inn_col].apply(normalize_id),
        'excel_ops_cnt': df[excel_ops_col].apply(normalize_num),
        'excel_retl_cnt': df[excel_retl_col].apply(normalize_num),
        'excel_term_cnt': df[excel_term_col].apply(normalize_num),
    }).dropna(subset=['inn'])

    # На случай дублей INN в Excel агрегируем до клиента
    x = x.groupby('inn', as_index=False).agg({
        'excel_ops_cnt': 'sum',
        'excel_retl_cnt': 'sum',
        'excel_term_cnt': 'sum',
    })

    excel_by_month[month_key] = x
    print(month_key, 'excel_unique_inn =', len(x))

## 1) Функции легких агрегатов из Озера (с узким периметром)

In [ ]:
def fetch_sa_active_base(month_start):
    with imp:
        return imp.fetch(f"""
            select
                cast(c.c_inn as string) as inn,
                cast(a.n_cmp_client as string) as n_cmp_client,
                cast(a.n_agr as string) as n_agr
            from ods_alpha.scd1_agreements a
            join ods_alpha.scd1_companies c
              on c.n_cmp = a.n_cmp_client
            where a.acq_class = 'SA'
              and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
              and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
              and c.c_inn is not null
            group by cast(c.c_inn as string), cast(a.n_cmp_client as string), cast(a.n_agr as string)
        """)

def fetch_trx_by_inn(base_df, month_start, month_end, step=500):
    if len(base_df) == 0:
        return pd.DataFrame(columns=['inn', 'lake_trx_cnt'])

    pairs = base_df[['inn', 'n_agr']].dropna().drop_duplicates()
    n_agr_list = sorted(pairs['n_agr'].astype(str).unique().tolist())
    out = []

    for chunk in split_chunks(n_agr_list, step=step):
        in_agr = to_sql_in_list(chunk)
        with imp:
            part = imp.fetch(f"""
                select
                    cast(ta.n_agr as string) as n_agr,
                    count(distinct t.n_trx) as trx_cnt
                from ods_alpha.scd1_trx_acq ta
                join ods_alpha.scd1_trx t
                  on t.n_trx = ta.n_trx
                where cast(ta.n_agr as string) in ({in_agr})
                  and to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
                  and t.c_trx_class = 'SA'
                  and t.c_trx_type = 'S01'
                  and t.c_nter is not null
                  and t.ods_deleted_flg <> '1'
                  and t.cf_trx_stat <> 'R'
                group by cast(ta.n_agr as string)
            """)
        out.append(part)

    trx_agr = pd.concat(out, ignore_index=True) if out else pd.DataFrame(columns=['n_agr', 'trx_cnt'])
    z = pairs.merge(trx_agr, on='n_agr', how='left')
    z['trx_cnt'] = z['trx_cnt'].fillna(0)
    res = z.groupby('inn', as_index=False)['trx_cnt'].sum().rename(columns={'trx_cnt': 'lake_trx_cnt'})
    return res

def fetch_retl_by_inn(base_df, step=500):
    if len(base_df) == 0:
        return pd.DataFrame(columns=['inn', 'lake_retl_cnt'])

    pairs = base_df[['inn', 'n_cmp_client']].dropna().drop_duplicates()
    cmp_list = sorted(pairs['n_cmp_client'].astype(str).unique().tolist())
    out = []
    for chunk in split_chunks(cmp_list, step=step):
        in_cmp = to_sql_in_list(chunk)
        with imp:
            part = imp.fetch(f"""
                select
                    cast(m.n_cmp as string) as n_cmp_client,
                    count(distinct cast(m.c_nmrc as string)) as retl_cnt
                from ods_alpha.scd1_merchants m
                where cast(m.n_cmp as string) in ({in_cmp})
                  and m.c_nmrc is not null
                group by cast(m.n_cmp as string)
            """)
        out.append(part)

    retl_cmp = pd.concat(out, ignore_index=True) if out else pd.DataFrame(columns=['n_cmp_client', 'retl_cnt'])
    z = pairs.merge(retl_cmp, on='n_cmp_client', how='left')
    z['retl_cnt'] = z['retl_cnt'].fillna(0)
    res = z.groupby('inn', as_index=False)['retl_cnt'].sum().rename(columns={'retl_cnt': 'lake_retl_cnt'})
    return res

def fetch_term_by_inn(base_df, month_start, month_end, step=500):
    if len(base_df) == 0:
        return pd.DataFrame(columns=['inn', 'lake_term_cnt'])

    pairs = base_df[['inn', 'n_cmp_client']].dropna().drop_duplicates()
    cmp_list = sorted(pairs['n_cmp_client'].astype(str).unique().tolist())
    out = []
    for chunk in split_chunks(cmp_list, step=step):
        in_cmp = to_sql_in_list(chunk)
        with imp:
            part = imp.fetch(f"""
                select
                    cast(m.n_cmp as string) as n_cmp_client,
                    count(distinct cast(t.c_nter as string)) as term_cnt
                from ods_alpha.scd1_merchants m
                join ods_alpha.scd1_pos_terminals t
                  on t.c_nmrc = m.c_nmrc
                where cast(m.n_cmp as string) in ({in_cmp})
                  and t.c_nter is not null
                  and cast(t.d_ter_install as date) <= cast('{month_end}' as date)
                  and (t.d_ter_close is null or cast(t.d_ter_close as date) >= cast('{month_start}' as date))
                group by cast(m.n_cmp as string)
            """)
        out.append(part)

    term_cmp = pd.concat(out, ignore_index=True) if out else pd.DataFrame(columns=['n_cmp_client', 'term_cnt'])
    z = pairs.merge(term_cmp, on='n_cmp_client', how='left')
    z['term_cnt'] = z['term_cnt'].fillna(0)
    res = z.groupby('inn', as_index=False)['term_cnt'].sum().rename(columns={'term_cnt': 'lake_term_cnt'})
    return res

## 2) Smoke-check периметра (контроль объема до тяжелых запросов)

In [ ]:
smoke_rows = []
base_by_month = {}

for month_key in month_files.keys():
    month_start, month_end = month_bounds(month_key)
    base_df = fetch_sa_active_base(month_start)
    base_by_month[month_key] = base_df

    smoke_rows.append({
        'month': month_key,
        'sa_active_rows': len(base_df),
        'sa_active_unique_inn': base_df['inn'].nunique() if len(base_df) else 0,
        'sa_active_unique_n_cmp': base_df['n_cmp_client'].nunique() if len(base_df) else 0,
        'sa_active_unique_n_agr': base_df['n_agr'].nunique() if len(base_df) else 0,
    })

smoke_df = pd.DataFrame(smoke_rows)
display(smoke_df)

## 3) Расчет агрегатов из Озера по месяцам

In [ ]:
lake_by_month = {}

for month_key in month_files.keys():
    month_start, month_end = month_bounds(month_key)
    base_df = base_by_month[month_key]

    trx_df = fetch_trx_by_inn(base_df, month_start, month_end, step=chunk_size)
    retl_df = fetch_retl_by_inn(base_df, step=chunk_size)
    term_df = fetch_term_by_inn(base_df, month_start, month_end, step=chunk_size)

    lake_df = (
        base_df[['inn']].drop_duplicates()
        .merge(trx_df, on='inn', how='left')
        .merge(retl_df, on='inn', how='left')
        .merge(term_df, on='inn', how='left')
    )

    for c in ['lake_trx_cnt', 'lake_retl_cnt', 'lake_term_cnt']:
        lake_df[c] = lake_df[c].fillna(0)

    lake_by_month[month_key] = lake_df
    print(month_key, 'lake_unique_inn =', len(lake_df))

## 4) Секции сравнения Excel vs Озеро

In [ ]:
compare_by_month = {}
summary_rows = []

for month_key in month_files.keys():
    x = excel_by_month[month_key].copy()
    l = lake_by_month[month_key].copy()

    cmp_df = x.merge(l, on='inn', how='outer', indicator=True)
    for c in ['excel_ops_cnt', 'excel_retl_cnt', 'excel_term_cnt', 'lake_trx_cnt', 'lake_retl_cnt', 'lake_term_cnt']:
        cmp_df[c] = cmp_df[c].fillna(0)

    cmp_df['trx_mismatch'] = (cmp_df['excel_ops_cnt'] != cmp_df['lake_trx_cnt'])
    cmp_df['retl_mismatch'] = (cmp_df['excel_retl_cnt'] != cmp_df['lake_retl_cnt'])
    cmp_df['term_mismatch'] = (cmp_df['excel_term_cnt'] != cmp_df['lake_term_cnt'])

    cmp_df['trx_diff'] = cmp_df['excel_ops_cnt'] - cmp_df['lake_trx_cnt']
    cmp_df['retl_diff'] = cmp_df['excel_retl_cnt'] - cmp_df['lake_retl_cnt']
    cmp_df['term_diff'] = cmp_df['excel_term_cnt'] - cmp_df['lake_term_cnt']

    compare_by_month[month_key] = cmp_df

    summary_rows.append({
        'month': month_key,
        'excel_unique_inn': x['inn'].nunique(),
        'lake_unique_inn': l['inn'].nunique(),
        'common_inn': int((cmp_df['_merge'] == 'both').sum()),
        'only_excel_inn': int((cmp_df['_merge'] == 'left_only').sum()),
        'only_lake_inn': int((cmp_df['_merge'] == 'right_only').sum()),
        'trx_mismatch_clients': int(cmp_df[cmp_df['_merge'] == 'both']['trx_mismatch'].sum()),
        'retl_mismatch_clients': int(cmp_df[cmp_df['_merge'] == 'both']['retl_mismatch'].sum()),
        'term_mismatch_clients': int(cmp_df[cmp_df['_merge'] == 'both']['term_mismatch'].sum()),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

## 5) Детализация несостыковок по секциям

In [ ]:
for month_key in month_files.keys():
    print('\n' + '='*100)
    print('MONTH:', month_key)
    cmp_df = compare_by_month[month_key]

    only_excel = cmp_df[cmp_df['_merge'] == 'left_only'][['inn', 'excel_ops_cnt', 'excel_retl_cnt', 'excel_term_cnt']].copy()
    only_lake = cmp_df[cmp_df['_merge'] == 'right_only'][['inn', 'lake_trx_cnt', 'lake_retl_cnt', 'lake_term_cnt']].copy()
    both = cmp_df[cmp_df['_merge'] == 'both'].copy()

    print('only_excel_inn:', len(only_excel))
    display(only_excel.head(100))

    print('only_lake_inn:', len(only_lake))
    display(only_lake.head(100))

    trx_mm = both[both['trx_mismatch']].sort_values('trx_diff', key=lambda s: s.abs(), ascending=False)
    retl_mm = both[both['retl_mismatch']].sort_values('retl_diff', key=lambda s: s.abs(), ascending=False)
    term_mm = both[both['term_mismatch']].sort_values('term_diff', key=lambda s: s.abs(), ascending=False)

    print('trx_mismatch_clients:', len(trx_mm))
    display(trx_mm[['inn', 'excel_ops_cnt', 'lake_trx_cnt', 'trx_diff']].head(100))

    print('retl_mismatch_clients:', len(retl_mm))
    display(retl_mm[['inn', 'excel_retl_cnt', 'lake_retl_cnt', 'retl_diff']].head(100))

    print('term_mismatch_clients:', len(term_mm))
    display(term_mm[['inn', 'excel_term_cnt', 'lake_term_cnt', 'term_diff']].head(100))